# CWS QC × residual-risk fusion — audited runner

This runner is the post-residual-risk decision layer. It fuses:

1. the audited CatBoost residual-risk score (`context_history` by default),
2. lenient / strict / ultra-strict CrowdQC-style flags,
3. calibrated OWS-reference residual evidence,
4. train-only station residual-history features,
5. explicit sunny daytime / low-wind radiation-or-siting evidence,
6. sign-aware LCZ / LC / elevation / building / LST / NDVI context evidence,

into action-oriented categories and paired raw vs policy-bias-corrected outputs.

The fused scores are **evidence scores**, not verified ground-truth probabilities of sensor failure.


In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 220)
pd.set_option("display.max_rows", 120)


## 1. Project setup

This follows the same pattern as the previous notebooks. Edit `SCRIPT_DIR` if the script is not in the current directory.


In [ ]:
USE_CONFIG_FILES = True
city = os.environ.get("QC_PROJECT_ID", "project_id")
SCRIPT_DIR = Path.cwd()
if not (SCRIPT_DIR / "qc_risk_fusion_audited.py").exists():

    SCRIPT_DIR = Path.cwd().parent

if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

PROJECT_DIR = Path(os.environ.get("QC_DATA_ROOT", Path.cwd() / "data")).expanduser().resolve() / city

if USE_CONFIG_FILES:
    try:
        cwd = os.getcwd()
        cwd_main = os.path.abspath(os.path.join(cwd, os.pardir))
        os.chdir(cwd_main)
        print("cwd_main:", cwd_main)

        import config_tool as cfm

        cwd_project = Path(cfm.cwd_data) / city
        os.chdir(cwd_project)
        print("cwd_project:", cwd_project)

        if str(cwd_project) not in sys.path:
            sys.path.insert(0, str(cwd_project))
        import config_project as cfp

        PROJECT_DIR = cwd_project
        RESULTS_DIR = Path(getattr(cfp, "cwd_results", PROJECT_DIR / "results"))
        DATA_DIR = Path(getattr(cfp, "cwd_data", PROJECT_DIR / "data"))
        QC_DIR = Path(getattr(cfp, "cwd_data_qc", PROJECT_DIR / "data" / "quality control"))
        year_span = f"{pd.to_datetime(cfp.first_date, dayfirst=True).year}-{pd.to_datetime(cfp.last_date, dayfirst=True).year}"
    except Exception as e:
        print("Config import failed; falling back to manual paths. Reason:", repr(e))
        USE_CONFIG_FILES = False

if not USE_CONFIG_FILES:
    PROJECT_DIR = Path(PROJECT_DIR)
    RESULTS_DIR = PROJECT_DIR / "results"
    DATA_DIR = PROJECT_DIR / "data"
    QC_DIR = PROJECT_DIR / "data" / "quality control"
    year_span = "2021-2021"

print("SCRIPT_DIR:", SCRIPT_DIR)
print("PROJECT_DIR:", PROJECT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("QC_DIR:", QC_DIR)
print("year_span:", year_span)


## 2. User-editable configuration

Use `context_history` as the main paper-facing model. Use `context_static_met` as a stricter ablation.


In [ ]:
os.chdir(cfm.cwd_scripts_preprocesing)


from qc_risk_fusion_audited import FusionConfig, run_fusion, load_dataframe_auto

reference_run_label = "catboost_corrected_ows_metadata_iteration01"
reference_method = "catboost"
calibration_mode = "time_train"

residual_risk_run_label = (
    f"{reference_run_label}__residual_risk_AUDITED__{reference_method}__{calibration_mode}"
)

risk_model_key = "context_static_met"
risk_model_name = f"catboost_{risk_model_key}"
prob_col = f"pred_ref_risk_prob_{risk_model_key}"

RESIDUAL_RISK_DIR = (
    RESULTS_DIR / "20_quality_control" / "qc_benchmark" / "residual_risk_audited" / residual_risk_run_label
)
RESIDUAL_RISK_MANIFEST = RESIDUAL_RISK_DIR / f"{city}_residual_risk_manifest.json"


SCORED_PATH = None
RELIABILITY_PATH = None


QC_LENIENT = None
QC_STRICT = None
QC_ULTRA_STRICT = None

fusion_run_label = f"{residual_risk_run_label}__qc_risk_fusion_AUDITED__{risk_model_key}"
FUSION_OUTPUT_BASE = RESULTS_DIR / "20_quality_control" / "qc_benchmark" / "qc_risk_fusion_audited"

print("RESIDUAL_RISK_DIR:", RESIDUAL_RISK_DIR)
print("RESIDUAL_RISK_MANIFEST:", RESIDUAL_RISK_MANIFEST)
print("FUSION_OUTPUT_BASE:", FUSION_OUTPUT_BASE)


## 3. Thresholds

These are policy thresholds. They should be reported as chosen operating points, not as universal truths.


In [ ]:
cfg = FusionConfig(
    city=city,
    scored_input=str(SCORED_PATH) if SCORED_PATH is not None else None,
    residual_risk_manifest=str(RESIDUAL_RISK_MANIFEST),
    residual_risk_dir=str(RESIDUAL_RISK_DIR),
    output_dir=str(FUSION_OUTPUT_BASE),
    run_label=fusion_run_label,
    overwrite=True,

    risk_model_key=risk_model_key,
    risk_model_name=risk_model_name,
    prob_col=prob_col,
    reliability_path=str(RELIABILITY_PATH) if RELIABILITY_PATH is not None else None,
    reliability_calibration_split="valid",
    use_reliability_calibration=True,

    qc_lenient=str(QC_LENIENT) if QC_LENIENT is not None else None,
    qc_strict=str(QC_STRICT) if QC_STRICT is not None else None,
    qc_ultra_strict=str(QC_ULTRA_STRICT) if QC_ULTRA_STRICT is not None else None,
    qc_dir=str(QC_DIR),
    auto_discover_qc=True,

    evaluation_split="test",


    resid_z_center=3.8,
    resid_z_slope=1.25,
    low_abs_z_for_valid=2.05,

    model_low_prob=0.05,
    model_high_prob=0.50,
    final_low_error_prob=0.15,
    final_moderate_error_prob=0.30,
    final_high_error_prob=0.70,

    reference_uncertainty_high=0.80,
    reference_uncertainty_moderate=0.55,
    bias_explained_high=0.55,
    bias_explained_moderate=0.35,
    microclimate_support_high=0.55,
    microclimate_support_moderate=0.35,
    environmental_directional_support_high=0.50,
    environmental_directional_support_moderate=0.30,

    radiation_bias_high=0.60,
    radiation_bias_moderate=0.40,
    radiation_low_wind_quantile=0.25,
    radiation_min_positive_resid_c=0.75,

    min_abs_expected_bias_c=0.50,

    add_station_bias_correction_preview=True,
    save_policy_output_tables=True,
    sample_rows_per_category=250,
)

cfg


## 4. Run audited fusion


In [ ]:
manifest = run_fusion(cfg)
print(json.dumps(manifest["outputs"], indent=2))


In [ ]:


qc_audit_quick = pd.read_csv(manifest["outputs"]["qc_audit_path"])
print("QC files used:")
print(json.dumps(manifest.get("qc_files", {}), indent=2))
print("\nQC read / merge audit:")
display(qc_audit_quick)


## 5. Load and inspect saved outputs


In [ ]:
out = manifest["outputs"]

fused = load_dataframe_auto(out["fused_path"])

raw_policy_output = None
bias_corrected_policy_output = None
if out.get("raw_policy_output_path"):
    raw_policy_output = load_dataframe_auto(out["raw_policy_output_path"])
if out.get("bias_corrected_policy_output_path"):
    bias_corrected_policy_output = load_dataframe_auto(out["bias_corrected_policy_output_path"])

category_summary = pd.read_csv(out["category_summary_path"])
action_summary = pd.read_csv(out["action_summary_path"])
method_comparison = pd.read_csv(out["method_comparison_path"])
correction_summary = pd.read_csv(out["correction_summary_path"])
feature_group_audit = pd.read_csv(out["feature_group_audit_path"])
station_network_audit = pd.read_csv(out["station_network_audit_path"])
qc_pattern_summary = pd.read_csv(out["qc_pattern_path"])
micro_component_audit = pd.read_csv(out["micro_component_path"])

print("Fused rows:", len(fused))
print("Fused columns:", len(fused.columns))
print("Output folder:", Path(out["fused_path"]).parent)
if raw_policy_output is not None:
    print("Raw policy output rows:", len(raw_policy_output))
if bias_corrected_policy_output is not None:
    print("Bias-corrected policy output rows:", len(bias_corrected_policy_output))

display(feature_group_audit)
display(station_network_audit.head(20))
display(category_summary)
display(action_summary)
display(method_comparison)
display(correction_summary)
display(qc_pattern_summary.head(20))
display(micro_component_audit)


## 6. Essential paper checks

The most important checks are:

- Are the metadata/context feature groups present with reasonable coverage?
- Did QC-rescue now use model/reference evidence **excluding QC**?
- How many rows fall into rescue / missed-risk / systemic-bias / radiation-siting / environmental-preserve categories?
- Does the raw policy output differ from the policy-bias-corrected output in expected ways?
- Does the correction preview reduce held-out residual error for bias-correctable candidates while preserving environmental-difference candidates?
- Does the conservative policy reduce residual risk while retaining a defensible number of stations?


In [ ]:

keep_rows = method_comparison[
    method_comparison["method"].isin([
        "raw_all_observed",
        "fusion_keep_conservative",
        "fusion_keep_microclimate",
        "fusion_bias_correctable_only",
        "fusion_remove_reject_transient_only",
    ])
].copy()
display(keep_rows)

display(correction_summary)

if raw_policy_output is not None and bias_corrected_policy_output is not None:
    product_compare = pd.DataFrame({
        "product": ["raw_no_bias_correction", "policy_bias_corrected"],
        "n_rows": [len(raw_policy_output), len(bias_corrected_policy_output)],
        "mean_product_residual": [
            raw_policy_output["cws_ref_resid_product"].mean() if "cws_ref_resid_product" in raw_policy_output else np.nan,
            bias_corrected_policy_output["cws_ref_resid_product"].mean() if "cws_ref_resid_product" in bias_corrected_policy_output else np.nan,
        ],
        "mae_product_residual": [
            raw_policy_output["cws_ref_resid_product"].abs().mean() if "cws_ref_resid_product" in raw_policy_output else np.nan,
            bias_corrected_policy_output["cws_ref_resid_product"].abs().mean() if "cws_ref_resid_product" in bias_corrected_policy_output else np.nan,
        ],
    })
    display(product_compare)


## 7. Plots


In [ ]:
if len(category_summary):
    plt.figure(figsize=(11, 5))
    plt.bar(category_summary["fusion_category"].astype(str), category_summary["fraction"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Fraction of evaluation rows")
    plt.title("QC × residual-risk fusion categories")
    plt.tight_layout()
    plt.show()

if len(method_comparison):
    plot_df = method_comparison.dropna(subset=["retention", "residual_mae"]).copy()
    plt.figure(figsize=(8, 5))
    plt.scatter(plot_df["retention"], plot_df["residual_mae"])
    for _, r in plot_df.iterrows():
        label = str(r["method"])
        if (
            label in {"raw_all_observed", "fusion_keep_conservative", "fusion_keep_microclimate"}
            or label.startswith("risk_")
            or label.startswith("fusion_reject_score")
        ):
            plt.annotate(label, (r["retention"], r["residual_mae"]), fontsize=8, alpha=0.8)
    plt.xlabel("Retention")
    plt.ylabel("MAE of CWS - OWS reference residual (°C)")
    plt.title("Retention vs residual error by data-use policy")
    plt.tight_layout()
    plt.show()

if len(correction_summary):
    plot_df = correction_summary.dropna(subset=["coverage", "residual_mae"]).copy()
    plt.figure(figsize=(8, 5))
    plt.scatter(plot_df["coverage"], plot_df["residual_mae"])
    for _, r in plot_df.iterrows():
        plt.annotate(str(r["method"]), (r["coverage"], r["residual_mae"]), fontsize=8, alpha=0.8)
    plt.xlabel("Coverage")
    plt.ylabel("MAE after selected correction policy (°C)")
    plt.title("Correction-preview: coverage vs residual error")
    plt.tight_layout()
    plt.show()


## 8. Interpretation notes

Recommended paper framing:

- `p_reference_model_issue_no_qc` = model/reference residual evidence excluding CrowdQC; this is used for QC-rescue.
- `p_any_quality_issue` = evidence of CWS–reference disagreement or QC concern.
- `p_reject_as_unexplained_transient_error` = evidence for removal, after downweighting persistent bias, reference uncertainty, radiation/siting evidence, and environmental support.
- `systemic_station_bias_correction_candidate` = persistent station-history bias candidate; bias-correct then use.
- `radiation_or_siting_bias_candidate` = sunny daytime / low-wind positive-residual candidate; context-bias-correct or downweight.
- `environmental_difference_candidate_preserve` = sign-aware LCZ/LST/elevation/NDVI/context preservation candidate; do not automatically correct away.
- `qc_flagged_low_reference_risk_rescue_candidate` = key evidence that ultra/strict QC can over-remove data.
- `qc_missed_probable_high_risk_observation` = key evidence that even strict QC can leave residual risk.
- The saved policy outputs provide paired `raw_no_bias_correction` and `policy_bias_corrected` products for comparison.
